# Module 04 -- Implied Volatility & the Smile

**Author: Djellal Djouad** -- CrossVol Research | [crossvol.com](https://crossvol.com) | ORCID [0009-0002-4911-1118](https://orcid.org/0009-0002-4911-1118)

Implied volatility is the market's consensus on future uncertainty, backed out from
option prices. It's the single number that, when plugged into BSM, reproduces the
observed market price. The fact that this number varies by strike (the "smile") and
by expiry (the "term structure") tells you that BSM is wrong -- but usefully wrong.

Every derivatives desk lives and breathes implied vol. It's the common language.
When a trader says "the 25-delta put is bid at 18", they mean 18% implied vol.
Not dollars, not euros -- vol.

*License: MIT with Educational Use Clause -- see LICENSE. Not trading advice.*


In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt


In [ ]:
# --- BSM building blocks ---

def bsm_price(S, K, T, r, sigma, option_type='call'):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == 'call':
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def bsm_vega(S, K, T, r, sigma):
    """Vega in absolute terms (not per vol point)."""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return S * norm.pdf(d1) * np.sqrt(T)


## Newton-Raphson IV Solver

We want to find $\sigma$ such that $BSM(\sigma) = C_{market}$.
Newton-Raphson is the standard approach because vega (the derivative of price
w.r.t. sigma) is available in closed form. Convergence is quadratic, so
5-6 iterations is usually enough.

I've seen production systems that use Brent's method as a fallback for edge cases
(deep OTM, near-zero vega), but for educational purposes Newton-Raphson is clean
and transparent.


In [ ]:
def implied_vol(market_price, S, K, T, r, option_type='call',
                tol=1e-8, max_iter=100, initial_guess=0.20):
    """
    Newton-Raphson solver for implied volatility.

    Returns NaN if it fails to converge -- don't silently return garbage.
    """
    sigma = initial_guess

    for _ in range(max_iter):
        price = bsm_price(S, K, T, r, sigma, option_type)
        v = bsm_vega(S, K, T, r, sigma)

        if v < 1e-12:
            # Vega near zero means we're in no-man's land (deep ITM/OTM or T~0).
            # Don't divide by zero. Bail.
            return np.nan

        diff = price - market_price
        if abs(diff) < tol:
            return sigma

        sigma = sigma - diff / v

        # Guard against negative vol or runaway
        if sigma <= 0:
            sigma = 0.001

    return np.nan  # did not converge


### Quick test: round-trip

Price an option at 25% vol, then back out the IV. Should recover 25%.


In [ ]:
S, K, T, r = 100, 100, 0.5, 0.05
test_vol = 0.25
test_price = bsm_price(S, K, T, r, test_vol, 'call')
recovered_vol = implied_vol(test_price, S, K, T, r, 'call')

print(f"Input vol:     {test_vol:.4f}")
print(f"BSM price:     {test_price:.4f}")
print(f"Recovered IV:  {recovered_vol:.4f}")
print(f"Error:         {abs(recovered_vol - test_vol):.2e}")


## Building a Smile from Market Prices

In reality, you'd pull option prices from a data vendor. Here we'll simulate
a realistic smile by generating prices from a known vol function, then back
out the IVs to confirm our solver works. In the next section, we'll look at
what creates this smile shape.


In [ ]:
# Simulate a smile: vol increases away from ATM, steeper on the downside (skew)
K_range = np.linspace(85, 115, 31)
T_smile = 0.25

def synthetic_smile(K, S=100, atm_vol=0.20, skew=-0.08, kurtosis=0.02):
    """Generate a realistic smile shape (log-moneyness parameterization)."""
    m = np.log(K / S)
    return atm_vol + skew * m + kurtosis * m**2

true_vols = synthetic_smile(K_range)

# Generate "market" prices from these vols
market_prices = np.array([
    bsm_price(100, k, T_smile, 0.05, v, 'call')
    for k, v in zip(K_range, true_vols)
])

# Now recover IVs using our solver
recovered_ivs = np.array([
    implied_vol(p, 100, k, T_smile, 0.05, 'call')
    for p, k in zip(market_prices, K_range)
])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(K_range, market_prices, 'o-', color='steelblue', markersize=4)
axes[0].set_title('Simulated Market Prices')
axes[0].set_xlabel('Strike')
axes[0].set_ylabel('Call Price')
axes[0].grid(True, alpha=0.3)

axes[1].plot(K_range, true_vols * 100, 'o-', label='True Vol', color='steelblue', markersize=4)
axes[1].plot(K_range, recovered_ivs * 100, 'x', label='Recovered IV', color='firebrick', markersize=7)
axes[1].set_title('Volatility Smile')
axes[1].set_xlabel('Strike')
axes[1].set_ylabel('Implied Vol (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Why Does the Smile Exist?

BSM assumes constant volatility. If that were true, you'd get a flat line --
the same IV for every strike. The smile (or smirk) tells you the market
disagrees with BSM's assumptions. Here's why:

1. **Fat tails / jumps.** Real returns have fatter tails than the normal
   distribution. Crashes happen more often than BSM predicts. OTM puts are
   priced higher (in vol terms) to reflect this.

2. **Supply and demand.** Portfolio managers buy OTM puts for protection.
   That demand pushes put prices (and thus put IVs) up. Dealers who sell
   those puts are short skew and charge for it.

3. **Leverage effect.** When stocks drop, leverage increases, which tends to
   increase volatility. This creates a negative correlation between returns
   and vol, which steepens the skew.

After the 1987 crash, the equity smile became a permanent smirk -- OTM puts
have never gone back to being "cheap" relative to ATM. The market learned.


## Smile Across Expirations

Short-dated smiles are steeper. Long-dated smiles are flatter. This is because
short-term, the probability of a big move is more binary -- either it happens
or it doesn't. Over longer horizons, things average out.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

tenors = [0.04, 0.125, 0.25, 0.5, 1.0]
labels = ['2W', '6W', '3M', '6M', '1Y']
colors = plt.colormaps['viridis'](np.linspace(0.1, 0.9, len(tenors)))

for T_i, label, color in zip(tenors, labels, colors):
    # Skew flattens with tenor (realistic behavior)
    skew_adj = -0.08 / np.sqrt(T_i / 0.25)
    kurt_adj = 0.02 / np.sqrt(T_i / 0.25)
    vols = synthetic_smile(K_range, skew=skew_adj, kurtosis=kurt_adj)
    ax.plot(K_range, vols * 100, label=label, color=color, lw=2)

ax.set_xlabel('Strike')
ax.set_ylabel('Implied Vol (%)')
ax.set_title('Volatility Smile by Expiration')
ax.legend(title='Tenor')
ax.grid(True, alpha=0.3)
ax.axvline(100, color='grey', ls='--', lw=0.7)
plt.tight_layout()
plt.show()


## Stress Scenario: Skew Steepening

When the market sells off, two things happen simultaneously:
- ATM vol jumps (the whole smile shifts up)
- The skew steepens (OTM puts get even more expensive relative to ATM)

This is the "smirk getting meaner" effect. If you're short skew (as most
market-making desks are, structurally), a crash hits you twice: vol up AND
skew blowout. I've lived through a few of these. The 2020 COVID crash was
textbook -- 25-delta put vol went from 18 to 65 in about a week on SPX.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Normal regime
normal_smile = synthetic_smile(K_range, atm_vol=0.18, skew=-0.06, kurtosis=0.015)
ax.plot(K_range, normal_smile * 100, label='Normal', lw=2, color='steelblue')

# Stress regime
stress_smile = synthetic_smile(K_range, atm_vol=0.32, skew=-0.18, kurtosis=0.04)
ax.plot(K_range, stress_smile * 100, label='Stress', lw=2, color='firebrick')

ax.set_xlabel('Strike')
ax.set_ylabel('Implied Vol (%)')
ax.set_title('Smile: Normal vs. Stress Regime')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axvline(100, color='grey', ls='--', lw=0.7)
plt.tight_layout()
plt.show()


## Practical Notes

- **Newton-Raphson can fail** for deep OTM options where vega is tiny. In production,
  use Brent's method or a hybrid (NR with Brent fallback).
- **Always sanity-check recovered IVs.** If you get 300% vol on a 6M equity option,
  something is wrong with your input price (stale quote, bad bid-ask, etc.).
- **The smile is information.** Don't treat it as noise. It tells you what risks
  the market is pricing in. Learning to read the smile is like learning to read
  an X-ray -- it takes practice, but it becomes second nature.

---

**Next:** [Module 05 -- Volatility Surface](05_vol_surface.py)

*Djellal Djouad -- CrossVol Research -- 2026*
